In [2]:
!pip install networkx

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ----- -------------------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from __future__ import annotations

import os
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

try:
    import yfinance as yf
except ImportError as exc:
    raise ImportError("Please install yfinance first: pip install yfinance") from exc

warnings.filterwarnings("ignore")

In [4]:
# 1. Configuration
START_DATE = "2018-01-01"
END_DATE = "2026-04-01"  # yfinance end date is exclusive; includes data through 2026-03-31 when available
P_LAGS = 5
EDGE_THRESHOLD = 1e-4
CV_SPLITS = 5
ROLLING_WINDOW = 500
ROLLING_STEP = 20
OUTPUT_DIR = "outputs_sparse_granger"

ASSET_CLASSES: Dict[str, List[str]] = {
    "Equity": ["SPY", "QQQ", "IWM", "EFA", "EEM"],
    "Bond": ["TLT", "IEF", "SHY", "LQD", "HYG"],
    "Commodity": ["GLD", "SLV", "USO", "UNG", "DBA"],
    "FX": ["UUP", "FXE", "FXY", "FXB", "FXC"],
    "Crypto": ["BTC-USD", "ETH-USD", "BNB-USD", "SOL-USD", "XRP-USD"],
}

TICKER_TO_CLASS = {ticker: cls for cls, tickers in ASSET_CLASSES.items() for ticker in tickers}
TICKERS = list(TICKER_TO_CLASS.keys())
ASSET_CLASS_ORDER = ["Equity", "Bond", "Commodity", "FX", "Crypto"]

REGIMES = {
    "Pre-COVID": ("2018-01-01", "2019-12-31"),
    "COVID Crisis": ("2020-02-01", "2020-06-30"),
    "Liquidity Boom": ("2020-07-01", "2021-12-31"),
    "Rate-Hike Shock": ("2022-01-01", "2023-06-30"),
    "Post-Tightening": ("2023-07-01", "2026-03-31"),
}


@dataclass
class NetworkResult:
    tickers: List[str]
    asset_classes: Dict[str, str]
    A_mats: List[pd.DataFrame]
    W: pd.DataFrame
    adjacency: pd.DataFrame
    metrics: pd.DataFrame
    class_matrix: pd.DataFrame
    density: float
    total_weighted_spillover: float
    edge_count: int

In [5]:
# 2. Data Download and Variable Construction
def ensure_output_dir() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "figures"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "tables"), exist_ok=True)


def download_prices(tickers: List[str], start: str, end: str) -> pd.DataFrame:
    """Download adjusted close prices from Yahoo Finance."""
    print("Downloading price data from Yahoo Finance...")
    raw = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,
        group_by="column",
        progress=True,
        threads=True,
    )

    if isinstance(raw.columns, pd.MultiIndex):
        if "Close" in raw.columns.get_level_values(0):
            prices = raw["Close"].copy()
        elif "Adj Close" in raw.columns.get_level_values(0):
            prices = raw["Adj Close"].copy()
        else:
            raise ValueError("Cannot find Close or Adj Close columns in downloaded data.")
    else:
        prices = raw[["Close"]].copy()
        prices.columns = tickers

    prices = prices.reindex(columns=tickers)
    prices = prices.dropna(how="all")
    prices.index = pd.to_datetime(prices.index)
    return prices


def clean_prices(prices: pd.DataFrame) -> pd.DataFrame:
    """Align all assets to common dates. Baseline uses inner join by dropping rows with missing values."""
    prices = prices.sort_index()
    prices = prices.replace([np.inf, -np.inf], np.nan)
    clean = prices.dropna(axis=0, how="any")
    print(f"Price data shape after inner-join cleaning: {clean.shape}")
    return clean


def compute_returns(prices: pd.DataFrame) -> pd.DataFrame:
    """Compute daily percentage log returns."""
    returns = 100.0 * np.log(prices / prices.shift(1))
    returns = returns.replace([np.inf, -np.inf], np.nan).dropna(how="any")
    return returns


def compute_volatility_proxy(returns: pd.DataFrame, proxy: str = "log_squared", c: float = 1e-6) -> pd.DataFrame:
    """Construct volatility proxy."""
    if proxy == "log_squared":
        vol = np.log(returns.pow(2) + c)
    elif proxy == "absolute":
        vol = returns.abs()
    elif proxy == "squared":
        vol = returns.pow(2)
    else:
        raise ValueError("proxy must be one of: log_squared, absolute, squared")
    vol = vol.replace([np.inf, -np.inf], np.nan).dropna(how="any")
    return vol


def standardize_df(df: pd.DataFrame) -> pd.DataFrame:
    scaler = StandardScaler()
    arr = scaler.fit_transform(df.values)
    return pd.DataFrame(arr, index=df.index, columns=df.columns)

In [6]:
# 3. Sparse VAR Estimation

def make_lagged_design(vol: pd.DataFrame, p: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Create lagged predictor matrix X and current response matrix Y."""
    X_parts = []
    for lag in range(1, p + 1):
        lagged = vol.shift(lag).copy()
        lagged.columns = [f"{col}_lag{lag}" for col in vol.columns]
        X_parts.append(lagged)
    X = pd.concat(X_parts, axis=1)
    Y = vol.copy()
    data = pd.concat([Y, X], axis=1).dropna(how="any")
    Y_clean = data[vol.columns]
    X_clean = data.drop(columns=vol.columns)
    return X_clean, Y_clean


def estimate_sparse_var(
    vol: pd.DataFrame,
    p: int = 5,
    edge_threshold: float = 1e-4,
    cv_splits: int = 5,
    model_type: str = "lasso",
) -> NetworkResult:
    """Estimate sparse VAR using equation-by-equation LASSO or Elastic Net."""
    vol_std = standardize_df(vol)
    tickers = list(vol_std.columns)
    n = len(tickers)

    X, Y = make_lagged_design(vol_std, p=p)
    # X is already based on standardized volatility, but standardizing predictors again improves numerical stability.
    x_scaler = StandardScaler()
    X_scaled = x_scaler.fit_transform(X.values)

    tscv = TimeSeriesSplit(n_splits=cv_splits)
    coef_cube = np.zeros((p, n, n))  # lag, receiver i, transmitter j

    for i, target in enumerate(tickers):
        y = Y[target].values
        if model_type == "lasso":
            model = LassoCV(
                cv=tscv,
                fit_intercept=True,
                max_iter=50000,
                n_alphas=100,
                random_state=42,
            )
        elif model_type == "elasticnet":
            model = ElasticNetCV(
                cv=tscv,
                fit_intercept=True,
                max_iter=50000,
                n_alphas=100,
                l1_ratio=[0.2, 0.5, 0.8, 0.95, 1.0],
                random_state=42,
            )
        else:
            raise ValueError("model_type must be lasso or elasticnet")

        model.fit(X_scaled, y)
        coef = model.coef_

        # X column order: lag1 all tickers, lag2 all tickers, ..., lagp all tickers
        for lag in range(p):
            start = lag * n
            end = (lag + 1) * n
            coef_cube[lag, i, :] = coef[start:end]

    A_mats = []
    for lag in range(p):
        A = pd.DataFrame(coef_cube[lag], index=tickers, columns=tickers)
        A_mats.append(A)

    # Weighted adjacency: W[i, j] = spillover from j to i; row receiver, column transmitter
    W_arr = np.sum(np.abs(coef_cube), axis=0)
    np.fill_diagonal(W_arr, 0.0)
    W = pd.DataFrame(W_arr, index=tickers, columns=tickers)

    adjacency = (W > edge_threshold).astype(int)
    density = adjacency.values.sum() / (n * (n - 1))
    edge_count = int(adjacency.values.sum())
    total_weighted_spillover = float(W.values.sum())

    metrics = compute_network_metrics(W, adjacency, TICKER_TO_CLASS)
    class_matrix = compute_asset_class_matrix(W, TICKER_TO_CLASS, ASSET_CLASS_ORDER)

    return NetworkResult(
        tickers=tickers,
        asset_classes=TICKER_TO_CLASS,
        A_mats=A_mats,
        W=W,
        adjacency=adjacency,
        metrics=metrics,
        class_matrix=class_matrix,
        density=density,
        total_weighted_spillover=total_weighted_spillover,
        edge_count=edge_count,
    )


# ============================================================
# 4. Network Metrics
# ============================================================

def compute_network_metrics(W: pd.DataFrame, adjacency: pd.DataFrame, ticker_to_class: Dict[str, str]) -> pd.DataFrame:
    """Compute asset-level directed network metrics.

    W[i, j] means j -> i.
    Thus out-strength for j is column sum of W[:, j].
    In-strength for j is row sum of W[j, :].
    """
    tickers = list(W.columns)
    out_degree = adjacency.sum(axis=0)
    in_degree = adjacency.sum(axis=1)
    out_strength = W.sum(axis=0)
    in_strength = W.sum(axis=1)
    net_strength = out_strength - in_strength

    metrics = pd.DataFrame({
        "AssetClass": [ticker_to_class[t] for t in tickers],
        "OutDegree": out_degree,
        "InDegree": in_degree,
        "OutStrength": out_strength,
        "InStrength": in_strength,
        "NetStrength": net_strength,
    }, index=tickers)
    metrics.index.name = "Ticker"
    metrics = metrics.sort_values("NetStrength", ascending=False)
    return metrics


def compute_asset_class_matrix(
    W: pd.DataFrame,
    ticker_to_class: Dict[str, str],
    class_order: List[str],
) -> pd.DataFrame:
    """Compute asset-class-level spillover matrix.

    Matrix entry [from_class, to_class] = sum spillover from assets in from_class to assets in to_class.
    """
    mat = pd.DataFrame(0.0, index=class_order, columns=class_order)
    tickers = list(W.columns)
    for transmitter in tickers:
        from_cls = ticker_to_class[transmitter]
        for receiver in tickers:
            to_cls = ticker_to_class[receiver]
            if transmitter == receiver:
                continue
            mat.loc[from_cls, to_cls] += W.loc[receiver, transmitter]
    return mat


def crypto_spillover_indices(class_matrix: pd.DataFrame) -> Dict[str, float]:
    crypto_to_trad = class_matrix.loc["Crypto", [c for c in class_matrix.columns if c != "Crypto"]].sum()
    trad_to_crypto = class_matrix.loc[[c for c in class_matrix.index if c != "Crypto"], "Crypto"].sum()
    return {
        "Crypto_to_Traditional": float(crypto_to_trad),
        "Traditional_to_Crypto": float(trad_to_crypto),
        "Crypto_Net": float(crypto_to_trad - trad_to_crypto),
    }


In [7]:
# 5. Tables

def descriptive_statistics(returns: pd.DataFrame, vol: pd.DataFrame) -> pd.DataFrame:
    stats = pd.DataFrame(index=returns.columns)
    stats["AssetClass"] = [TICKER_TO_CLASS[t] for t in returns.columns]
    stats["MeanReturn"] = returns.mean()
    stats["StdReturn"] = returns.std()
    stats["SkewReturn"] = returns.skew()
    stats["KurtReturn"] = returns.kurtosis()
    stats["MeanVolProxy"] = vol.mean()
    stats["MaxVolProxy"] = vol.max()
    stats["Observations"] = returns.count()
    return stats


def network_summary(result: NetworkResult, label: str = "Full Sample") -> pd.DataFrame:
    top_transmitter = result.metrics.sort_values("OutStrength", ascending=False).index[0]
    top_receiver = result.metrics.sort_values("InStrength", ascending=False).index[0]
    within_edges = 0
    cross_edges = 0
    for i in result.tickers:
        for j in result.tickers:
            if i == j:
                continue
            if result.adjacency.loc[i, j] == 1:
                if TICKER_TO_CLASS[i] == TICKER_TO_CLASS[j]:
                    within_edges += 1
                else:
                    cross_edges += 1
    crypto_idx = crypto_spillover_indices(result.class_matrix)
    summary = pd.DataFrame({
        "Sample": [label],
        "NumberOfEdges": [result.edge_count],
        "NetworkDensity": [result.density],
        "TotalWeightedSpillover": [result.total_weighted_spillover],
        "WithinClassEdges": [within_edges],
        "CrossClassEdges": [cross_edges],
        "TopTransmitter_OutStrength": [top_transmitter],
        "TopReceiver_InStrength": [top_receiver],
        "Crypto_to_Traditional": [crypto_idx["Crypto_to_Traditional"]],
        "Traditional_to_Crypto": [crypto_idx["Traditional_to_Crypto"]],
        "Crypto_Net": [crypto_idx["Crypto_Net"]],
    })
    return summary

In [8]:
# 6. Plotting Functions

def savefig(name: str) -> None:
    path = os.path.join(OUTPUT_DIR, "figures", name)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {path}")


def plot_asset_class_volatility(vol: pd.DataFrame) -> None:
    class_avg = pd.DataFrame(index=vol.index)
    for cls, tickers in ASSET_CLASSES.items():
        available = [t for t in tickers if t in vol.columns]
        class_avg[cls] = vol[available].mean(axis=1)

    plt.figure(figsize=(12, 6))
    for cls in ASSET_CLASS_ORDER:
        plt.plot(class_avg.index, class_avg[cls], label=cls, linewidth=1.2)
    plt.title("Time Series of Cross-Asset Volatility Proxies")
    plt.xlabel("Date")
    plt.ylabel("Average log-squared return volatility proxy")
    plt.legend()
    savefig("figure1_asset_class_volatility.png")


def plot_network(result: NetworkResult, title: str, filename: str, top_quantile: float = 0.80) -> None:
    W = result.W.copy()
    weights = W.values.flatten()
    weights = weights[weights > 0]
    if len(weights) == 0:
        print("No edges to plot.")
        return
    cutoff = np.quantile(weights, top_quantile)

    G = nx.DiGraph()
    for t in result.tickers:
        G.add_node(t, asset_class=TICKER_TO_CLASS[t])

    for receiver in result.tickers:
        for transmitter in result.tickers:
            if receiver == transmitter:
                continue
            weight = W.loc[receiver, transmitter]
            if weight >= cutoff and weight > EDGE_THRESHOLD:
                G.add_edge(transmitter, receiver, weight=weight)

    pos = nx.spring_layout(G, seed=42, k=0.8)
    class_to_int = {cls: idx for idx, cls in enumerate(ASSET_CLASS_ORDER)}
    node_colors = [class_to_int[TICKER_TO_CLASS[n]] for n in G.nodes()]
    node_sizes = []
    metrics = result.metrics.reindex(list(G.nodes()))
    for n in G.nodes():
        val = max(metrics.loc[n, "OutStrength"], 0.01)
        node_sizes.append(500 + 1000 * val / (metrics["OutStrength"].max() + 1e-12))

    edge_widths = [1.0 + 4.0 * G[u][v]["weight"] / max(weights) for u, v in G.edges()]

    plt.figure(figsize=(12, 9))
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, cmap=plt.cm.tab10, alpha=0.85)
    nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.45, arrows=True, arrowsize=12)
    nx.draw_networkx_labels(G, pos, font_size=9)
    plt.title(title)
    plt.axis("off")
    savefig(filename)


def plot_heatmap(mat: pd.DataFrame, title: str, filename: str) -> None:
    plt.figure(figsize=(8, 6))
    im = plt.imshow(mat.values, aspect="auto")
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.xticks(range(len(mat.columns)), mat.columns, rotation=45, ha="right")
    plt.yticks(range(len(mat.index)), mat.index)
    plt.title(title)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            plt.text(j, i, f"{mat.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
    plt.xlabel("To asset class")
    plt.ylabel("From asset class")
    savefig(filename)


def plot_net_transmitter_ranking(metrics: pd.DataFrame) -> None:
    plot_df = metrics.sort_values("NetStrength", ascending=True)
    plt.figure(figsize=(10, 8))
    plt.barh(plot_df.index, plot_df["NetStrength"])
    plt.axvline(0, linewidth=1)
    plt.title("Net Volatility Transmitters and Receivers")
    plt.xlabel("NetStrength = OutStrength - InStrength")
    plt.ylabel("Asset")
    savefig("figure4_net_transmitter_ranking.png")


def plot_regime_density(regime_summary: pd.DataFrame) -> None:
    plt.figure(figsize=(10, 5))
    plt.bar(regime_summary["Sample"], regime_summary["NetworkDensity"])
    plt.title("Network Density across Market Regimes")
    plt.xlabel("Regime")
    plt.ylabel("Network density")
    plt.xticks(rotation=30, ha="right")
    savefig("figure5_regime_network_density.png")


def plot_rolling_density(rolling_df: pd.DataFrame) -> None:
    plt.figure(figsize=(12, 5))
    plt.plot(pd.to_datetime(rolling_df["EndDate"]), rolling_df["Density"], linewidth=1.5)
    plt.title("Rolling Network Density of Cross-Asset Volatility Spillovers")
    plt.xlabel("Window end date")
    plt.ylabel("Network density")
    savefig("figure7_rolling_network_density.png")


def plot_rolling_crypto_spillovers(rolling_df: pd.DataFrame) -> None:
    dates = pd.to_datetime(rolling_df["EndDate"])
    plt.figure(figsize=(12, 6))
    plt.plot(dates, rolling_df["Crypto_to_Traditional"], label="Crypto to Traditional", linewidth=1.5)
    plt.plot(dates, rolling_df["Traditional_to_Crypto"], label="Traditional to Crypto", linewidth=1.5)
    plt.plot(dates, rolling_df["Crypto_Net"], label="Crypto Net", linewidth=1.5)
    plt.axhline(0, linewidth=1)
    plt.title("Rolling Crypto-to-Traditional and Traditional-to-Crypto Volatility Spillovers")
    plt.xlabel("Window end date")
    plt.ylabel("Weighted spillover")
    plt.legend()
    savefig("figure8_rolling_crypto_spillover_index.png")


In [9]:
# 7. Regime and Rolling Estimation
# ============================================================

def run_regime_analysis(vol: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, NetworkResult]]:
    summaries = []
    results = {}
    for name, (start, end) in REGIMES.items():
        sub = vol.loc[start:end].copy()
        print(f"Estimating regime: {name}, observations={len(sub)}")
        if len(sub) < 150:
            print(f"Skipping {name}: not enough observations.")
            continue
        res = estimate_sparse_var(sub, p=P_LAGS, edge_threshold=EDGE_THRESHOLD, cv_splits=CV_SPLITS)
        results[name] = res
        summaries.append(network_summary(res, label=name))
        plot_heatmap(res.class_matrix, f"Asset-Class Spillover Matrix: {name}", f"figure6_heatmap_{name.replace(' ', '_')}.png")
    regime_summary = pd.concat(summaries, ignore_index=True)
    return regime_summary, results


def run_rolling_analysis(vol: pd.DataFrame) -> pd.DataFrame:
    rows = []
    dates = vol.index
    total = len(vol)
    for start_idx in range(0, total - ROLLING_WINDOW + 1, ROLLING_STEP):
        end_idx = start_idx + ROLLING_WINDOW
        window = vol.iloc[start_idx:end_idx].copy()
        start_date = window.index[0]
        end_date = window.index[-1]
        print(f"Rolling window: {start_date.date()} to {end_date.date()}")
        res = estimate_sparse_var(window, p=P_LAGS, edge_threshold=EDGE_THRESHOLD, cv_splits=CV_SPLITS)
        cidx = crypto_spillover_indices(res.class_matrix)
        top_net = res.metrics.sort_values("NetStrength", ascending=False).index[0]
        top_net_class = TICKER_TO_CLASS[top_net]
        rows.append({
            "StartDate": start_date,
            "EndDate": end_date,
            "Density": res.density,
            "TotalWeightedSpillover": res.total_weighted_spillover,
            "Crypto_to_Traditional": cidx["Crypto_to_Traditional"],
            "Traditional_to_Crypto": cidx["Traditional_to_Crypto"],
            "Crypto_Net": cidx["Crypto_Net"],
            "Equity_to_Bond": res.class_matrix.loc["Equity", "Bond"],
            "Bond_to_Equity": res.class_matrix.loc["Bond", "Equity"],
            "TopNetTransmitter": top_net,
            "TopNetTransmitterClass": top_net_class,
        })
    return pd.DataFrame(rows)


In [10]:
# 8. Main Script

def main() -> None:
    ensure_output_dir()

    # Data
    prices_raw = download_prices(TICKERS, START_DATE, END_DATE)
    prices = clean_prices(prices_raw)
    returns = compute_returns(prices)
    vol = compute_volatility_proxy(returns, proxy="log_squared", c=1e-6)

    # Align returns and vol
    common_index = returns.index.intersection(vol.index)
    returns = returns.loc[common_index]
    vol = vol.loc[common_index]

    # Save raw processed data
    prices.to_csv(os.path.join(OUTPUT_DIR, "tables", "prices_clean.csv"))
    returns.to_csv(os.path.join(OUTPUT_DIR, "tables", "returns.csv"))
    vol.to_csv(os.path.join(OUTPUT_DIR, "tables", "volatility_proxy_log_squared.csv"))

    # Table 1: Asset list
    asset_list = pd.DataFrame({
        "Ticker": TICKERS,
        "AssetClass": [TICKER_TO_CLASS[t] for t in TICKERS],
    })
    asset_list.to_csv(os.path.join(OUTPUT_DIR, "tables", "table1_asset_list.csv"), index=False)

    # Table 2: Descriptive statistics
    desc = descriptive_statistics(returns, vol)
    desc.to_csv(os.path.join(OUTPUT_DIR, "tables", "table2_descriptive_statistics.csv"))

    # Figure 1
    plot_asset_class_volatility(vol)

    # Full-sample network
    print("Estimating full-sample sparse VAR network...")
    full_res = estimate_sparse_var(vol, p=P_LAGS, edge_threshold=EDGE_THRESHOLD, cv_splits=CV_SPLITS)
    full_res.W.to_csv(os.path.join(OUTPUT_DIR, "tables", "full_sample_weighted_adjacency_W_receiver_by_transmitter.csv"))
    full_res.adjacency.to_csv(os.path.join(OUTPUT_DIR, "tables", "full_sample_binary_adjacency_receiver_by_transmitter.csv"))
    full_res.metrics.to_csv(os.path.join(OUTPUT_DIR, "tables", "table4_full_sample_network_metrics.csv"))
    full_res.class_matrix.to_csv(os.path.join(OUTPUT_DIR, "tables", "figure3_asset_class_spillover_matrix.csv"))
    network_summary(full_res, "Full Sample").to_csv(os.path.join(OUTPUT_DIR, "tables", "table3_full_sample_network_summary.csv"), index=False)

    # Figures 2-4
    plot_network(full_res, "Full-Sample Sparse Granger-Causal Volatility Spillover Network", "figure2_full_sample_network.png")
    plot_heatmap(full_res.class_matrix, "Asset-Class-Level Volatility Spillover Matrix", "figure3_asset_class_spillover_heatmap.png")
    plot_net_transmitter_ranking(full_res.metrics)

    # Regime analysis
    regime_summary, regime_results = run_regime_analysis(vol)
    regime_summary.to_csv(os.path.join(OUTPUT_DIR, "tables", "table5_regime_comparison.csv"), index=False)
    plot_regime_density(regime_summary)

    # Rolling-window analysis
    print("Running rolling-window analysis. This may take some time...")
    rolling_df = run_rolling_analysis(vol)
    rolling_df.to_csv(os.path.join(OUTPUT_DIR, "tables", "rolling_window_results.csv"), index=False)
    plot_rolling_density(rolling_df)
    plot_rolling_crypto_spillovers(rolling_df)

    # Robustness checks: alternative volatility proxy and lag orders
    print("Running robustness checks...")
    robustness_rows = []
    specs = [
        ("Baseline", "log_squared", 5),
        ("AbsoluteReturn", "absolute", 5),
        ("Lag3", "log_squared", 3),
        ("Lag10", "log_squared", 10),
    ]
    for spec_name, proxy, lag in specs:
        vol_spec = compute_volatility_proxy(returns, proxy=proxy, c=1e-6)
        res_spec = estimate_sparse_var(vol_spec, p=lag, edge_threshold=EDGE_THRESHOLD, cv_splits=CV_SPLITS)
        cidx = crypto_spillover_indices(res_spec.class_matrix)
        top_transmitter = res_spec.metrics.sort_values("OutStrength", ascending=False).index[0]
        robustness_rows.append({
            "Specification": spec_name,
            "VolatilityProxy": proxy,
            "LagOrder": lag,
            "MainTransmitter_OutStrength": top_transmitter,
            "CryptoNetSign": np.sign(cidx["Crypto_Net"]),
            "CryptoNetValue": cidx["Crypto_Net"],
            "Density": res_spec.density,
            "TotalWeightedSpillover": res_spec.total_weighted_spillover,
        })

    # Robustness: excluding crypto
    non_crypto_tickers = [t for t in TICKERS if TICKER_TO_CLASS[t] != "Crypto"]
    vol_no_crypto = vol[non_crypto_tickers].copy()
    # Temporarily estimate with subset; metrics use global ticker class map safely.
    res_no_crypto = estimate_sparse_var(vol_no_crypto, p=P_LAGS, edge_threshold=EDGE_THRESHOLD, cv_splits=CV_SPLITS)
    robustness_rows.append({
        "Specification": "ExcludingCrypto",
        "VolatilityProxy": "log_squared",
        "LagOrder": P_LAGS,
        "MainTransmitter_OutStrength": res_no_crypto.metrics.sort_values("OutStrength", ascending=False).index[0],
        "CryptoNetSign": np.nan,
        "CryptoNetValue": np.nan,
        "Density": res_no_crypto.density,
        "TotalWeightedSpillover": res_no_crypto.total_weighted_spillover,
    })

    robustness = pd.DataFrame(robustness_rows)
    robustness.to_csv(os.path.join(OUTPUT_DIR, "tables", "table6_robustness_checks.csv"), index=False)

    print("All outputs saved in:", OUTPUT_DIR)


if __name__ == "__main__":
    main()

[*********************100%***********************]  25 of 25 completed


Price data shape after inner-join cleaning: (1500, 25)
Saved figure: outputs_sparse_granger\figures\figure1_asset_class_volatility.png
Estimating full-sample sparse VAR network...
Saved figure: outputs_sparse_granger\figures\figure2_full_sample_network.png
Saved figure: outputs_sparse_granger\figures\figure3_asset_class_spillover_heatmap.png
Saved figure: outputs_sparse_granger\figures\figure4_net_transmitter_ranking.png
Estimating regime: Pre-COVID, observations=0
Skipping Pre-COVID: not enough observations.
Estimating regime: COVID Crisis, observations=55
Skipping COVID Crisis: not enough observations.
Estimating regime: Liquidity Boom, observations=380
Saved figure: outputs_sparse_granger\figures\figure6_heatmap_Liquidity_Boom.png
Estimating regime: Rate-Hike Shock, observations=375
Saved figure: outputs_sparse_granger\figures\figure6_heatmap_Rate-Hike_Shock.png
Estimating regime: Post-Tightening, observations=689
Saved figure: outputs_sparse_granger\figures\figure6_heatmap_Post-Tig